# Class 04 · Part 1: Getting several precipitation datasets

How much does it rain on Earth? There is no single answer: different groups **estimate** precipitation in
different ways, and their estimates differ. In this class you compare two of them. In this notebook you learn

1. the three ways to estimate precipitation, and what **FROGS** is,
2. **where** FROGS is and how its files are organised,
3. how to **download** one year of a product with a function, and
4. that FROGS products share the **same grid**, which makes comparing easy.

In part 2, `class04_compare_datasets.ipynb`, you compare them with maps and time series.

## 1. Three ways to estimate precipitation

| Way | How | Strong | Weak |
|---|---|---|---|
| **Rain gauges** | a bucket at a station, as in class 02 | direct measurement | only where there are stations: few over the ocean |
| **Satellites** | measure clouds and rain from space | everywhere, every day | indirect: the rain is computed from what the satellite sees |
| **Reanalysis** | a weather model combined with observations (class 07) | complete in space and time | depends on the model |

Many products combine them, e.g. satellites corrected with rain gauges.

**FROGS** (*Frequent Rainfall Observations on GridS*, Roca et al., 2019) collects more than 50 of these
products, all on the **same grid of 1° × 1°**, with one value per day. Because the grid is the same, you
can subtract two products directly.

## 2. Where is the data?

The list of products, with their periods and references, is on the FROGS page:
**https://frogs.ipsl.fr/download-data/**

The files are on a data server of IPSL. Open its catalog in a new tab and click through it:
**https://thredds-x.ipsl.fr/thredds/catalog/FROGs/catalog.html**

There is one **folder per product**, and in it **one file per year**. In this class we use 2020 of:

| Product (folder name) | What | File of 2020 |
|---|---|---|
| `GPCP_CDR_v1.3_yes_enforced` | GPCP: satellites combined with rain gauges | `GPCP_CDR.1DD.2020.nc` |
| `IMERG_V07B_FC` | IMERG: satellites, calibrated with rain gauges | `IMERG_V07B_FC.1DD.2020.nc` |

Each file is about **95 MB**.

## 3. Download with a function

The course function `download_frogs` downloads one year of one product:

In [ ]:
from utils import download_frogs

help(download_frogs)

Its code. The steps are the same as in classes 02 and 03; what is new:

- The list of files of a product is in its `catalog.xml`, a list the server makes for programs.
- File names differ between products, but the **year is always the last number** in the name.
  `re.findall(r"\d+", name)` finds all the numbers in a name; `[-1]` takes the last one.
- Each product gets its own folder in `data`, e.g. `data/IMERG_V07B_FC/`.
- The download itself is done by `download_file`, which you met in class 03.

In [ ]:
import inspect

print(inspect.getsource(download_frogs))

Now download 2020 of both products (about half a minute each, only the first time):

In [ ]:
gpcp_file = download_frogs("GPCP_CDR_v1.3_yes_enforced", 2020)
imerg_file = download_frogs("IMERG_V07B_FC", 2020)

print(gpcp_file)
print(imerg_file)

## 4. The same grid

Open both. Each has one variable, `rain`, in mm/day.

In [ ]:
import xarray as xr

gpcp = xr.open_dataset(gpcp_file)["rain"]
imerg = xr.open_dataset(imerg_file)["rain"]
gpcp

In [ ]:
print("GPCP: ", gpcp.attrs["long_name"])
print("IMERG:", imerg.attrs["long_name"])
print("units:", gpcp.attrs["units"], imerg.attrs["units"])

Are the grids the same? `==` compares the latitudes value by value, `.all()` checks that all are equal:

In [ ]:
print("same latitudes: ", bool((gpcp.lat == imerg.lat).all()))
print("same longitudes:", bool((gpcp.lon == imerg.lon).all()))
print("same days:      ", bool((gpcp.time == imerg.time).all()))

So one day of both, side by side. `plt.subplots(1, 2)` makes a figure with two plots next to each other:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
gpcp.sel(time="2020-07-15").plot(ax=axes[0], vmax=30, cmap="Blues")
axes[0].set_title("GPCP, 15 July 2020")
imerg.sel(time="2020-07-15").plot(ax=axes[1], vmax=30, cmap="Blues")
axes[1].set_title("IMERG, 15 July 2020")
plt.show()

Similar, but not the same. In part 2 you find out how different they are, and where.

GPCP has some missing values (about 3 % of the days and cells); xarray skips them in means.

## 5. Exercise

1. Look at the list of products in the catalog (section 2) and choose a **third** one with data in 2020,
   e.g. `ERA5` (a reanalysis) or `CMORPH_v1.0_CRT` (satellites).
2. Download its 2020 with `download_frogs` and open it.
3. Check: is it on the same grid? What does its `long_name` say?

In [ ]:
# 1. and 2. download and open a third product

In [ ]:
# 3. the same grid?

## Next

Open **`class04_compare_datasets.ipynb`**: maps, the difference between the products, and time series.